## Setup

In [1]:
#online for colab environment
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/USA
from fairness_metric_utils import *
from penalty_utils import *

/content/drive/MyDrive/USA


In [3]:
%cd /content/drive/MyDrive/USA/AdultCensus

/content/drive/MyDrive/USA/AdultCensus


In [4]:
dataset_path = 'adult-preprocessed-2race-2age-2edu.csv'
df=pd.read_csv(dataset_path)

fair_metrics=['FPP', 'FPR', 'FPN', 'PPE'] #'GFA', 'PPA', 'FPR', 'PPE', 'FPA', 'OAE', 'EOP', 'FNP', 'FPP'
protected_attributes =['age', 'sex', 'race', 'edu']
mapping= {
    'age':{
        0: 'young',
        1: 'adult'
    },
    'edu':{
        0: 'low-edu',
        1: 'high-edu'
    },
    'sex':{
        0: 'female',
        1: 'male'
    },
    'race':{
        0: 'Amer-Black-Other',
        1: 'Asian-White'
    }
}
feature_cols= df.columns
target_variable = 'income'
target_variable_labels= ['0','1']
df.head()

,age,edu,marital.status,relationship,race,sex,capital.gain,capital.loss,hours.per.week,income
0,1,0,6,1,1,0,0,4356,40,0
1,1,0,6,1,1,0,0,4356,18,0
2,1,1,6,4,0,0,0,4356,40,0
3,1,0,0,4,1,0,0,3900,40,0
4,1,1,5,3,1,0,0,3900,40,0


## Compute penalities

In [14]:
def compute_predictions_validation(df, target_variable, sensible_attribute, target, target_variable_labels=['0','1']):
  Y = df[target_variable]
  X = df.drop(target_variable, axis=1)
  X_train, X_test_orig, y_train, y_test_orig = train_test_split(X, Y, test_size=0.7, random_state=1)
  X_val, X_test, y_val, y_test = train_test_split(X_test_orig, y_test_orig, test_size=0.5, random_state=1)
  sensible_indexes=df[sensible_attribute].loc[list(X_val.index)]
  model = RandomForestClassifier(random_state = 1234).fit(X_train,y_train)
  #feature_importance(model, X, X.columns)
  y_pred_val = model.predict(X_val)
  cm =confusion_matrix(y_val, y_pred_val, labels=target_variable_labels)
  #performance_metrics(y_test, y_pred)
  return sensible_indexes, X_train, y_train, X_val, X_test, y_val, y_test, y_pred_val

In [19]:
def get_attribute_analysis_validation(df, target_variable, sensible_attribute, fair_metrics, dataset_path, mapping, target_variable_labels=['0','1']):
  sensible_indexes={}
  y_pred_val= []
  y_val= []
  y_test= []
  differences = {}
  ratios= {}
  sensible_indexes, X_train, y_train, X_val, X_test, y_val, y_test, y_pred_val = compute_predictions_validation(df, target_variable, sensible_attribute, target_variable_labels)
  cm_dict={}
  fairness_metrics_dict={}
  cm_dict= compute_cm_group(df, sensible_attribute, sensible_indexes, y_pred_val, y_val, X_val, target_variable_labels)
  for m in fair_metrics:
    fairness_metrics_dict[m]= compute_fairness_metrics(cm_dict, m, sensible_attribute, mapping, dataset_path)
    differences[m], ratios[m] = get_max_min(fairness_metrics_dict[m])
  return fairness_metrics_dict, differences, ratios, X_train, y_train, X_val, X_test, y_val, y_test, y_pred_val

In [20]:
pairs_dict= {}
fairness_metrics_dict= {}
differences = {}
ratios= {}
X_train = {}
y_train = {}
X_val = {}
X_test = {}
y_val = {}
y_test = {}
y_pred_val = {}
s= []
for i in range(1,len(protected_attributes)+1):
  pairs_dict[i] = ['-'.join(pair) for pair in combinations(protected_attributes, i)]
#print(pairs_dict)

for i in range(1, len(protected_attributes)+1):
  df=pd.read_csv(dataset_path)
  for sensible_attribute in pairs_dict[i]:
    s = sensible_attribute.split('-')
    print(s)
    if len(s)>1:
      df=pd.read_csv(dataset_path)
      df[sensible_attribute] = reduce(lambda x, y: x.astype(str) + y.astype(str), [df[col] for col in s])
      df = df.drop(columns=s)
    fairness_metrics_dict[sensible_attribute], differences[sensible_attribute], ratios[sensible_attribute], X_train[sensible_attribute], y_train[sensible_attribute], X_val[sensible_attribute], X_test[sensible_attribute], y_val[sensible_attribute], y_test[sensible_attribute], y_pred_val[sensible_attribute] = get_attribute_analysis_validation(df, target_variable, sensible_attribute, fair_metrics, dataset_path, mapping,target_variable_labels)

['age']
FPP {1: 0.11130971725706858, 0: 0.02861503243037009}
Max value: 0.111 (Key: 1)
Min value: 0.029 (Key: 0)
Max-min: 0.082
Max-min: 0.261
FPR {1: 0.31581373905025356, 0: 0.410958904109589}
Max value: 0.411 (Key: 0)
Min value: 0.316 (Key: 1)
Max-min: 0.095
Max-min: 0.769
FPN {1: 0.5131086142322098, 0: 0.29069767441860467}
Max value: 0.513 (Key: 1)
Min value: 0.291 (Key: 0)
Max-min: 0.222
Max-min: 0.567
PPE {1: 0.17039800995024876, 0: 0.03218193520703712}
Max value: 0.17 (Key: 1)
Min value: 0.032 (Key: 0)
Max-min: 0.138
Max-min: 0.188
['sex']
FPP {0: 0.03938673010837959, 1: 0.09010902403783003}
Max value: 0.09 (Key: 1)
Min value: 0.039 (Key: 0)
Max-min: 0.051
Max-min: 0.433
FPR {0: 0.3952254641909814, 1: 0.31803430690774226}
Max value: 0.395 (Key: 0)
Min value: 0.318 (Key: 1)
Max-min: 0.077
Max-min: 0.805
FPN {0: 0.4613003095975232, 1: 0.4489528795811518}
Max value: 0.461 (Key: 0)
Min value: 0.449 (Key: 1)
Max-min: 0.012
Max-min: 0.974
PPE {0: 0.04406980183377699, 1: 0.1294339622641

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [23]:
def harmonic_mean_2(a,b):
  return (2)/((1/a)+(1/b))

def harmonic_mean_3(a,b,c):
  return (3)/((1/a)+(1/b)+(1/c))

def harmonic_mean_4(a,b,c,d):
  return (4)/((1/a)+(1/b)+(1/c)+(1/d))

def compute_penalty(actual_values, predicted_values):
  penalties= {}
  for k in actual_values.keys():
    penalties[k] = penalty_percentage(actual_values[k], predicted_values[k])
  return penalties

def penalty_percentage(actual_value, predicted_value):
  return ((predicted_value-actual_value)*100)/predicted_value

def actual_predicted_values_2(fairness_metrics_dict, df, s1, s2, m):
  actual_values= {}
  predicted_values= {}
  s3 = str(s1)+'-'+str(s2)
  for i in range(0,df[s1].nunique()):
    for j in range(0,df[s2].nunique()):
      if i in fairness_metrics_dict[s1][m] and j in fairness_metrics_dict[s2][m]:
        a = fairness_metrics_dict[s1][m][i]
        b = fairness_metrics_dict[s2][m][j]
        k=str(i)+str(j)
        if k in fairness_metrics_dict[s3][m]:
          c = fairness_metrics_dict[s3][m][k]
        else:
          c = 0
        actual_values[k]= c
        predicted_values[k]= harmonic_mean_2(a,b)
  return actual_values, predicted_values

def actual_predicted_values_3(fairness_metrics_dict, df, s1, s2, s3, m):
  actual_values= {}
  predicted_values= {}
  s4 = str(s1)+'-'+str(s2)+'-'+str(s3)
  for i in range(0,df[s1].nunique()):
    for j in range(0,df[s2].nunique()):
      for k in range(0,df[s3].nunique()):
        if i in fairness_metrics_dict[s1][m] and j in fairness_metrics_dict[s2][m] and k in fairness_metrics_dict[s3][m]:
          a = fairness_metrics_dict[s1][m][i]
          b = fairness_metrics_dict[s2][m][j]
          c = fairness_metrics_dict[s3][m][k]
          w=str(i)+str(j)+str(k)
          if w in fairness_metrics_dict[s4][m]:
            d = fairness_metrics_dict[s4][m][w]
          else:
            d = 0
          actual_values[w]= d
          predicted_values[w]= harmonic_mean_3(a,b,c)
  return actual_values, predicted_values

def actual_predicted_values_4(fairness_metrics_dict, df, s1, s2, s3, s4, m):
  actual_values= {}
  predicted_values= {}
  s5 = str(s1)+'-'+str(s2)+'-'+str(s3)+'-'+str(s4)
  for i in range(0,df[s1].nunique()):
    for j in range(0,df[s2].nunique()):
      for k in range(0,df[s3].nunique()):
        for l in range(0,df[s4].nunique()):
          if i in fairness_metrics_dict[s1][m] and j in fairness_metrics_dict[s2][m] and k in fairness_metrics_dict[s3][m] and l in fairness_metrics_dict[s4][m]:
            a = fairness_metrics_dict[s1][m][i]
            b = fairness_metrics_dict[s2][m][j]
            c = fairness_metrics_dict[s3][m][k]
            d = fairness_metrics_dict[s4][m][l]
            w=str(i)+str(j)+str(k)+str(l)
            if w in fairness_metrics_dict[s5][m]:
              e = fairness_metrics_dict[s5][m][w]
            else:
              e= 0
            actual_values[w]= e
            predicted_values[w]= harmonic_mean_4(a,b,c,d)
  return actual_values, predicted_values

values= {}
penalties_across_metrics= {}
for m in fair_metrics:
  print(m)
  df=pd.read_csv(dataset_path)
  actual_values= {}
  predicted_values= {}
  penalties= {}
  for i in range(1, len(protected_attributes)+1):
    for sensible_attribute in pairs_dict[i]:
      s = sensible_attribute.split('-')
      #print(sensible_attribute)
      if len(s)==2:
        s1, s2 = sensible_attribute.split('-')
        actual_values[sensible_attribute], predicted_values[sensible_attribute] = actual_predicted_values_2(fairness_metrics_dict, df,s1, s2, m)
        print(s1, s2, actual_values[sensible_attribute], predicted_values[sensible_attribute])
        penalties[sensible_attribute] = compute_penalty(actual_values[sensible_attribute], predicted_values[sensible_attribute])
        print(penalties[sensible_attribute])
      elif len(s)==3:
        s1, s2, s3 = sensible_attribute.split('-')
        actual_values[sensible_attribute], predicted_values[sensible_attribute] = actual_predicted_values_3(fairness_metrics_dict, df,s1, s2, s3, m)
        print(s1, s2, s3, actual_values[sensible_attribute], predicted_values[sensible_attribute])
        penalties[sensible_attribute] = compute_penalty(actual_values[sensible_attribute], predicted_values[sensible_attribute])
        print(penalties[sensible_attribute])
      elif len(s)==4:
        s1, s2, s3, s4 = sensible_attribute.split('-')
        actual_values[sensible_attribute], predicted_values[sensible_attribute] = actual_predicted_values_4(fairness_metrics_dict, df,s1, s2, s3, s4, m)
        print(s1, s2, s3, s4, actual_values[sensible_attribute], predicted_values[sensible_attribute])
        penalties[sensible_attribute] = compute_penalty(actual_values[sensible_attribute], predicted_values[sensible_attribute])
        print(penalties[sensible_attribute])
      values[m]= [actual_values, predicted_values]
      penalties_across_metrics[m]= penalties
print(penalties)

#print(values['FPP'])


FPP
age sex {'00': 0.025693730729701953, '01': 0.030643203883495146, '10': 0.05117038649972782, '11': 0.13551077136900624} {'00': 0.03314774550836925, '01': 0.04343639733708291, '10': 0.05818485928088839, '11': 0.09959361202914328}
{'00': 22.48724510324626, '01': 29.452703810372977, '10': 12.055494965276246, '11': -36.06371795146139}
age race {'00': 0.009584664536741214, '01': 0.03054592720970537, '10': 0.030215827338129497, '11': 0.12108444770104415} {'00': 0.02484521715290946, '01': 0.04215259238443164, '10': 0.03667324809780025, '11': 0.09309278700778038}
{'00': 61.42249641952187, '01': 27.534878682842287, '10': 17.60798700581443, '11': -30.068560189227462}
age edu {'00': 0.006672226855713094, '01': 0.04571026722925457, '10': 0.035571687840290384, '11': 0.17358046484260076} {'00': 0.025076683772114725, '01': 0.04585315527024703, '10': 0.0371798093930051, '11': 0.11328380038209164}
{'00': 73.39270648245518, '01': 0.31162095639943155, '10': 4.3252549675987995, '11': -53.22620203165523

## Selection of the level/subgroups

In [13]:
fairness_metrics_dict['age-sex-race-edu']['FPN']

{'1010': 0.06818181818181818,
 '1001': 0.0625,
 '1011': 0.32989690721649484,
 '0010': 0.21739130434782608,
 '1110': 0.07553956834532374,
 '1111': 0.793036750483559,
 '0111': 0.19831223628691982,
 '0101': 0,
 '0110': 0.06666666666666667,
 '1100': 0.09523809523809523,
 '0001': 0,
 '0011': 0.20754716981132076,
 '1000': 0,
 '0100': 0.25,
 '1101': 0.28205128205128205}

In [28]:
penalties_across_metrics['FPN']['age-sex-race-edu']

{'0000': 100.0,
 '0001': 33.42976912923107,
 '0010': 19.680614587413768,
 '0011': -43.0335252035829,
 '0100': -92.5142297655837,
 '0101': 100.0,
 '0110': 63.01616185723786,
 '0111': 34.15256445276619,
 '1000': 1.068403729845261,
 '1001': -12.409428566722791,
 '1010': -49.888978287884775,
 '1011': 24.789378339258743,
 '1100': 13.06222818290159,
 '1101': 68.34649996257538,
 '1110': 55.53833832378653,
 '1111': -58.0124526940887}

## Apply reweight

In [29]:
"""
Compute sample weights based on intersectional penalties for collapsed string groups.

Parameters:
  df (pd.DataFrame): Dataset with true labels, predictions, and group column.
  y_true_col (str): Column name of ground truth labels.
  y_pred_col (str): Column name of model predictions.
  group_col (str): Column name of the combined group (e.g., "sex-race").
  penalties (dict): Mapping from group strings (e.g., "femaleBlack") to penalty scores.
  lambda_ (float): Reweighting strength.
  focus_on (str): Type of error to apply reweighting on.
  normalize_counts (bool): Whether to normalize pattern counts.

Returns:
  pd.Series: Sample weights.
"""
def compute_sample_weights_flat_group(df, y_true_col, y_pred_col, group_col, penalties, lambda_=1.0, focus_on="fp"):

    weights = np.ones(len(df))  # Initialize weights to 1
    for i in range(len(df)):
        # Get the group for this row (sex-race)
        group = df.iloc[i][group_col]

        # Get the predicted and actual values for this row
        y_true = df.iloc[i][y_true_col]
        y_pred = df.iloc[i][y_pred_col]

        # Apply conditions for False Positives (or False Negatives, depending on focus)
        if focus_on == "fp" and y_true == 0 and y_pred == 1:
            # False positive — compute penalty and weight
            penalty = penalties.get(group, 0)
            weight = 1 + lambda_ * (penalty / 100)  # Scale the penalty
            weight = 1 + lambda_ * (penalty / 100)

            # Debugging output: print row index, group, penalty, and computed weight
            print(f"Row {i}, group={group}, penalty={penalty:.2f}, weight={weight:.2f}")

            # Assign the computed weight
            weights[i] = weight

        elif focus_on == "fn" and y_true == 1 and y_pred == 0:
            # False negative — compute penalty and weight (similar logic as above)
            penalty = penalties.get(group, 0)
            weight = 1 + lambda_ * (penalty / 100)
            weights[i] = weight

        # Optional: add more conditions for other focuses like "tn" or "tp" if needed

    return weights


In [31]:
sensible_attribute = 'age-sex-race-edu'
m='FPN'
s = sensible_attribute.split('-')
df=X_val[sensible_attribute].copy()
df["y_true"]= y_val[sensible_attribute]
df["y_pred"]= y_pred_val[sensible_attribute]

weights = compute_sample_weights_flat_group(
    df=df,
    y_true_col="y_true",
    y_pred_col="y_pred",
    group_col=sensible_attribute,
    penalties=penalties_across_metrics[m][sensible_attribute],
    lambda_=2.0,
    focus_on="fp"
)

print(weights)

Row 0, group=1111, penalty=-58.01, weight=-0.16
Row 23, group=1111, penalty=-58.01, weight=-0.16
Row 42, group=1111, penalty=-58.01, weight=-0.16
Row 59, group=1111, penalty=-58.01, weight=-0.16
Row 70, group=0111, penalty=34.15, weight=1.68
Row 72, group=1110, penalty=55.54, weight=2.11
Row 82, group=1111, penalty=-58.01, weight=-0.16
Row 108, group=1111, penalty=-58.01, weight=-0.16
Row 115, group=1110, penalty=55.54, weight=2.11
Row 124, group=1111, penalty=-58.01, weight=-0.16
Row 143, group=1011, penalty=24.79, weight=1.50
Row 151, group=1111, penalty=-58.01, weight=-0.16
Row 233, group=1111, penalty=-58.01, weight=-0.16
Row 240, group=1011, penalty=24.79, weight=1.50
Row 262, group=1111, penalty=-58.01, weight=-0.16
Row 267, group=1111, penalty=-58.01, weight=-0.16
Row 281, group=1111, penalty=-58.01, weight=-0.16
Row 285, group=1111, penalty=-58.01, weight=-0.16
Row 305, group=1111, penalty=-58.01, weight=-0.16
Row 326, group=1111, penalty=-58.01, weight=-0.16
Row 341, group=111